# This ain't your grandma's Python

There have been important changes in the Python ecosystem the past several years. You can improve your code by incorporating some of those changes. This workshop covers changes in

- [Managing Python environments](#section-1-managing-python-environments)
- [Creating dynamic text](#section-2-creating-dynamic-text)
- [Manipulating filepaths](#section-3-manipulating-filepaths)
- [Structuring data](#section-4-structuring-data)

## Section 1: Managing Python environments

### Problem

You wrote a script for other people.

It uses `tomllib`, introduced in Python 3.11.

It uses `fiona`, which:

- Is not included in the default ArcGIS Pro Python environment.
- Depends on having `GDAL` installed


How do you make sure they can run it?

### Exercise: Use `uv` and `pixi` to manage Python environments

Objectives:
- Use pyproject.toml to ensure the environment is always documented.
- Use a lock file to ensure dependency versions are always pinned.
- Improve discoverability by storing environments with projects.
- Improve efficiency by caching packages.

1. In the bash terminal, run the following command to install `uv`:

    ```bash
    curl -LsSf https://astral.sh/uv/install.sh | sh
    ```

1. In the bash terminal, run the following command to create a `uv` environment based on the pyproject.toml file in the workspace:

    ```bash
    uv sync
    ```

1. Note how long it took for the environment to build.

1. Click **Select Kernel**, then click **Python Environments**.

1. Click **modern-python**, which is the name of the environment set in the pyproject.toml file.

1. Execute the code cell below.

In [ ]:
import tomllib


5. Open pyproject.toml and determine which part is responsible for setting the Python version used in the environment.

5. Review the dependencies in the pyproject.toml file and determine if the `fiona` package is included.

5. Open the uv.lock file, which includes all the dependencies of the dependencies specified in the pyproject.toml file, and determine if `fiona` was installed implicitly.

5. In the bash terminal, run the following command to add `fiona` to the environment.

    ```bash
    uv add fiona
    ```

    That didn't work because `uv` can only add Python packages. The system GDAL that `fiona` needs isn't a Python package. While `uv` is great in many ways, it is not the right tool for this use case.

5. Delete the .venv directory and uv.lock file

5. In the bash terminal, run the following command to install `pixi`:

    ```bash
    curl -fsSL https://pixi.sh/install.sh | sh
    ```
5. In the bash terminal, run the following command to activate the `pixi` environment

    ```bash
    pixi shell
    ```

5. In the bash terminal, run the following command to add `fiona` to the environment:

    ```bash
    pixi add fiona
    ```

    This works because `pixi`, unlike `uv`, can use conda channels to get non-Python dependencies. Because geospatial projects often rely on these dependencies, `pixi` is often a better choice than `uv`.

    Because the remainder of this workshop does not require any non-Python dependencies, we will use the `uv` environment, which is typically easier to use than `pixi`.


5. In the bash terminal, run `exit` to close the `pixi` shell.

5. In the bash terminal, run `uv sync` again to rebuild the `uv` environment.

5. Compare how long the environment took to build this time to the original build.




## Section 2: Creating dynamic text

### Problem

Your code needs to make a string like: 

`"A Canada lynx was sighted at 48.8° N, 93.2° W"`

You have these variables whose values aren't known until runtime:

- `species`
- `lat`
- `lon`

How do you do it?

### Exercise: Use f-strings and t-strings to create dynamic text

Objectives:

- Use any expression, not just variable names, to generate dynamic text.
- Format values using the format specifier mini-language.
- Employ an appropriate strategy for updating existing code that creates dynamic text.
- Evaluate the use of t-strings and tag functions

The cell below has an f-string with the correct syntax. But it doesn't work. Fix it. 

In [ ]:
def growth_rate(new, old):
    return (new - old) / old

text = f"The population growth rate in {location} was {growth_rate(pop2025, pop2024)} last year."
print(text)

The cell below has an f-string that uses a [format specifier](https://docs.python.org/3/tutorial/inputoutput.html#formatted-string-literals) to represent the value as a percentage, not a decimal rate. Make it use 1 decimal place, not 2.

In [ ]:
text = f"The population growth rate in {location} was {growth_rate(pop2025, pop2024):.2%} last year."
print(text)

The cell below uses older style string `format` method to create dynamic text. Update the code cell so that it also prints out the growth rate.

In [ ]:
text1 = "The population in {} changed by {} people last year.".format(location, pop2025 - pop2024)
print(text1)

The cell below defines a t-string. Run the code and determine how useful the t-string is for your typical use cases.

In [ ]:
text = t"The population growth rate in {location} was {growth_rate(pop2025, pop2024):.2%} last year."
print(text)

Tag functions are used to parse t-strings. The code for such a function to print out a representation of the t-string is partially written below. The partial function only processes the dynamic parts of the t-string (the interpolations). Finish the tag function. 

While working on the tag function, think about whether you want to write such functions for your code, or whether you want libraries to write tag functions so you can pass them a t-string that "just works".

In [ ]:
from string.templatelib import Interpolation

def tag(tstring):
    substrings = []
    for part in tstring:

        # Debugging line to see what the parts of the t-string are
        print(part)

        # Populate the substrings list with the interpolated parts of the t-string
        if isinstance(part, Interpolation):
            substrings.append(str(part.value))

        # Populate the substrings list with the literal parts of the t-string

    return "".join(substrings)

tag(text)

## Section 3: Manipulating filepaths

### Problem

You have a web service that gets data from a set of partitioned GeoParquet files

The service URL should be based on the filepath needed for that query. For example:

- Filepath: `/data/country=Bangladesh/year=1997/buildings.parquet`
- URL: `https://api.example.com/buildings?country=Bangladesh&year=1997`

How do you get the right URL given a filepath?

### Exercise: Use `pathlib` to manipulate filepaths

Objectives:

- Leverage `pathlib` syntax for common tasks with paths and files.
- Differentiate abstract path manipulation from operations that actually touch the filesystem.
- Use a `Path` object instead of a filepath string as an argument to other functions.
- Leverage well-named methods to make code more self-documenting.

All the files listed below are supposed to be in the `C:\data` directory. Fix the code to so that it prints out the path, including the parent `data` directory.

In [50]:
from pathlib import Path

files = ["counties.txt", "states.txt", "countries.txt"]
directory = Path(r"C:\data")

for file in files:
    ...

Use the [`read_text`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.read_text) method to see the contents of `counties.txt`.

The code below reads data from `counties.txt` into a `pandas` DataFrame object. Rewrite the code to use a `Path` object instead of the `filepath` string.

In [33]:
import pandas as pd

filepath = "data/counties.txt"

df = pd.read_csv(filepath)


Because `counties.txt` is formatted as a csv, it would be better if the file extension reflected that. Rename the file to `counties.csv` using the [`rename`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.rename) and [`with_suffix`](https://docs.python.org/3/library/pathlib.html#pathlib.PurePath.with_suffix) methods.

## Section 4: Structuring data

### Problem

You have a Python API for updating parcel values.

The API expects a particular format:

In [22]:
correct = {"propertyId": "0123456789", "value": 251482.17}

Users also sometimes submit problematic values, like:

In [ ]:
missing_field = {"value": 251482.17}
wrong_types = {"propertyId": 1234567890, "value": "324398.71"}
uncoercable = {
    "propertyId": ["0123456789", "1234567890"],
    "value": [251482.17, 324398.71],
}

How do you:

- Ensure the submission has the right keys and value types
- Reject the submission only when impossible to parse

### Exercise: Use Pydantic v2 models to structure data

Objectives:

- Represent a single table row as a model.
- Enforce type checking.
- Use dot notation to access values.

The code below partially creates a Pydantic model for the data in `counties.csv`. Finish the model by defining the correct type for the `county` field.

In [84]:
from pydantic import BaseModel

class CountyModel(BaseModel):
    population: int

The code below doesn't work. Fix it.

In [ ]:
county = "Olmstead"
population = "166,000"

CountyModel(county=county, population=population)

ValidationError: 1 validation error for CountyModel
population
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='166,000', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing

The code below reads data from a csv, then tries to create a `CountyModel` object for each row. But it doesn't work. 

Fix the code so that it correctly creates the `CountyModel` objects. Then add code so that it prints out the population of each county.

In [ ]:
from csv import DictReader

with open("data/counties.csv") as f:
    csv_reader = DictReader(f)
    for row in csv_reader:
        county = CountyModel.model_validate(row, strict=True)